# Load libraries

In [ ]:
import os
import gc
import time
import warnings
import numpy as np
import cudf 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import gbvpp_feat_eng as feats

# from tqdm.notebook import tqdm
from tqdm import tqdm
from collections import Counter

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler

import torch

warnings.filterwarnings("ignore")
NUM_WORKERS = os.cpu_count()

# Read in data

In [ ]:
DATA_PATH = "/kaggle/input/ventilator-pressure-prediction/"

sub = pd.read_csv(DATA_PATH + 'sample_submission.csv')
df_train = pd.read_csv(DATA_PATH + 'train.csv')
df_test = pd.read_csv(DATA_PATH + 'test.csv')

# df = df_train[df_train['breath_id'] < 5].reset_index(drop=True)

In [ ]:
baseCols = [col for col in df_train.columns.tolist() if col not in ['pressure']]
len(baseCols)

In [ ]:
print(f'Length of TRAIN dataset: {len(df_train)}')
print(f'Length of TEST dataset: {len(df_test)}')
print('')
print(f'Number of breaths in train dataset: {df_train["breath_id"].nunique()}')
print(f'Number of breaths in test dataset: {df_test["breath_id"].nunique()}')
print(f'The number of observations for each breath: {df_train["breath_id"].value_counts().reset_index()["breath_id"].unique()[0]}')

# Feature Engineering

In [ ]:
df_train.dtypes

In [ ]:
fcols = df_train.select_dtypes('float').columns
icols = df_train.select_dtypes('integer').columns

# df_train[fcols] = df_train[fcols].apply(pd.to_numeric, downcast='float')
# df_train[icols] = df_train[icols].apply(pd.to_numeric, downcast='integer')

df_train[fcols] = df_train[fcols].astype(np.float32)
df_train[icols] = df_train[icols].astype(np.int32)

fcols = df_test.select_dtypes('float').columns
icols = df_test.select_dtypes('integer').columns

# df_test[fcols] = df_test[fcols].apply(pd.to_numeric, downcast='float')
# df_test[icols] = df_test[icols].apply(pd.to_numeric, downcast='integer')

df_test[fcols] = df_test[fcols].astype(np.float32)
df_test[icols] = df_test[icols].astype(np.int32)

print(df_train.dtypes)
print(df_test.dtypes)

In [ ]:
print(df_train.shape)
print(df_test.shape)

start = time.time()

# Add lag diff features
df_train = feats.features_lagdiff(df_train)
df_test = feats.features_lagdiff(df_test)

end = time.time()
print("Lag diff features completed")
print(df_train.shape)
print(df_test.shape)
print(f"Time elapsed: {end-start}")

df_train = cudf.from_pandas(df_train)
df_test = cudf.from_pandas(df_test)

# Add rolling features (lag)
df_train = feats.features_roll1(df_train)
df_test = feats.features_roll1(df_test)

end = time.time()
print("Roll1 features completed")
print(df_train.shape)
print(df_test.shape)
print(f"Time elapsed: {end-start}")

# Add rolling features (lead)
df_train = feats.features_roll2(df_train)
df_test = feats.features_roll2(df_test)

end = time.time()
print("Roll2 features completed")
print(df_train.shape)
print(df_test.shape)
print(f"Time elapsed: {end-start}")

# Add groupby features
df_train = feats.features_group(df_train)
df_test = feats.features_group(df_test)

end = time.time()
print("Group features completed")
print(df_train.shape)
print(df_test.shape)
print(f"Time elapsed: {end-start}")

In [ ]:
# df_train = df_train.to_pandas()
# df_test = df_test.to_pandas()

df_train.to_feather("df_train.feather")
df_test.to_feather("df_test.feather")

In [ ]:
# df_train['breath_steps'] = df_train[['breath_id','R']].groupby('breath_id')['R'].cumcount().reset_index(drop=True)+1
# df_train['R__C'] = df_train["R"].astype(str) + '__' + df_train["C"].astype(str)

In [ ]:
# df_train = df_train.to_pandas()
# df_test = df_test.to_pandas()

In [ ]:
# # Add dummy features
# df_train = feats.features_dummy(df_train)
# df_test = feats.features_dummy(df_test)

# end = time.time()
# print("Dummy features completed")
# print(df_train.shape)
# print(df_test.shape)
# print(f"Time elapsed: {end-start}")

In [ ]:
# results = gdf.groupby(["ID", "column_name"]).apply_grouped(cumcount,
#                                incols=['column_name'],
#                                outcols=dict(cumcount=np.int32))

# Testing

https://www.kaggle.com/andradaolteanu/answer-correctness-rapids-xgb-lgbm/notebook#1.-Feature-Engineering

In [ ]:
# df = df_train[df_train['breath_id'] < 5].reset_index(drop=True)

# df = cudf.from_pandas(df)

# # df_train['area'] = df_train['time_step'] * df_train['u_in']

In [ ]:
# # Reverse roll solved
# temp = df[["breath_id",'id', "u_in"]].iloc[::-1].reset_index(drop=True).groupby(["breath_id"]).rolling(window=5, min_periods=1)["u_in"].mean().iloc[::-1].reset_index(drop=False)
# temp['index'] = len(temp)-temp['index']
# temp['breath_id'] = temp['index'].applymap(lambda x: (x//80)+1 if (x%80)!=0 else x//80)

In [ ]:
# temp1 = df[['R','u_in','u_out']].groupby(['R','u_out']).mean()["u_in"].reset_index().rename(columns={'u_in':'u_in_grp_R_uout_mean'})
# temp2 = df[['R','u_in','u_out']].groupby(['R','u_out']).max()["u_in"].reset_index().rename(columns={'u_in':'u_in_grp_R_uout_max'})

# temp = temp1.merge(temp2, on=['R','u_out'], how="left")

# df = df.merge(temp, on=['R','u_out'], how="left")

# df['u_in_grp_R_uout_mean_diff'] = df["u_in"] - df["u_in_grp_R_uout_mean"]
# df['u_in_grp_R_uout_max_diff'] = df["u_in"] - df["u_in_grp_R_uout_max"]